In [4]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
#Pega os dados do seges
from request_seges import Seges as sg
from request_seges import Login
from urllib.parse import urlparse
import urllib3

import requests

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

session = requests.Session()
usuario = '10631094776'
etapa = 0 # significa que é o trimestre (0-1ºTrimestre, 1-2ºTrimestre...)
senha = usuario


login = Login("https://seges.sedu.es.gov.br")

session_logada, base_url = login.autenticar(usuario, senha)
seges = sg(session_logada, etapa, base_url)

# pega links das turmas
url = 'https://seges.sedu.es.gov.br/avaliacao_modo_avancados/turmas'
lancar_notas = seges.get_minhas_turmas(url)

# pega minhas notas
listagem_avaliacao = seges.get_links_minhas_notas(lancar_notas['href'])
listagem_avaliacao.tail()
'''
output
classroom é a turma
discipline_id é o tipo de diciplina
stage_id é o código do trimestre
'''

minhas_turmas = (listagem_avaliacao.drop_duplicates(subset='classroom_id').reset_index(drop=True)) # só serve para pegar o nome dos alunos e fim
meus_alunos = seges.get_alunos_por_turma(listagem_avaliacao['href'].to_list())
minhas_avaliacoes = seges.get_avaliacoes(listagem_avaliacao['href'])
listagem_avaliacao["classroom_id"] = listagem_avaliacao["classroom_id"].astype(int)

notas = seges.get_notas(listagem_avaliacao['href'])


In [6]:
meus_alunos[meus_alunos['nome'] == 'GUILHERME DE MATOS CAMILO']

,turma,classroom_id,discipline_id,aluno_id,numero,nome,condicao,classroom_evaluation_id,avaliacao_nome
1296,3ªV01-EM-ENE,36378,257667,916763,28,GUILHERME DE MATOS CAMILO,active,1762542,PROVA
1297,3ªV01-EM-ENE,36378,257667,916763,28,GUILHERME DE MATOS CAMILO,active,1762566,TRABALHOS EM GRUPOS
1298,3ªV01-EM-ENE,36378,257667,916763,28,GUILHERME DE MATOS CAMILO,active,1762553,TRABALHOS INDIVIDUAIS
1299,3ªV01-EM-ENE,36378,257667,916763,28,GUILHERME DE MATOS CAMILO,active,1762532,LISTAS DE EXERCICIOS
1300,3ªV01-EM-ENE,36378,257667,916763,28,GUILHERME DE MATOS CAMILO,active,1910162,INTERDISCIPLINAR


In [ ]:
import pandas as pd
import unicodedata

# ========= FUNÇÃO PRA LIMPAR NOME =========
def limpar_nome(nome):
    if pd.isna(nome):
        return None
    nome = str(nome).strip().upper()
    return unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('ASCII')


# ========= CAMINHO DO ARQUIVO =========
caminho = "input/nota.xlsx"

# ========= LER TODAS AS SHEETS =========
dfs = pd.read_excel(caminho, sheet_name=None)

lista_dfs = []

# ========= LOOP NAS ABAS =========
for nome_aba, df in dfs.items():
    try:
        # Garantir cópia
        df = df.copy()

        # Pegar até 4 colunas
        df = df.iloc[:, :4]

        # Detectar número de colunas
        n_cols = df.shape[1]

        if n_cols == 4:
            df.columns = ["id", "nome", "nota_1", "nota_2"]
        elif n_cols == 3:
            df.columns = ["id", "nome", "nota_1"]
            df["nota_2"] = None
        else:
            print(f"Aba {nome_aba} ignorada (colunas insuficientes)")
            continue

        # ========= LIMPEZA =========
        df["nome"] = df["nome"].apply(limpar_nome)

        df["nota_1"] = pd.to_numeric(df["nota_1"], errors="coerce")
        df["nota_2"] = pd.to_numeric(df["nota_2"], errors="coerce")

        # ========= REGRA DE NEGÓCIO =========
        df["nota_final"] = df[["nota_1", "nota_2"]].max(axis=1)

        # ========= GUARDAR TURMA DA SHEET =========
        df["turma_sheet"] = nome_aba

        # ========= LIMPEZA FINAL =========
        df = df.dropna(subset=["nome"])

        # ========= SELECIONAR COLUNAS =========
        lista_dfs.append(df[["nome", "nota_final", "turma_sheet"]])

    except Exception as e:
        print(f"Erro na aba {nome_aba}: {e}")


# ========= CONCATENAR =========
if lista_dfs:
    df_suja = pd.concat(lista_dfs, ignore_index=True)
else:
    df_suja = pd.DataFrame(columns=["nome", "nota_final", "turma_sheet"])



In [ ]:
df_merge = df_suja.merge(meus_alunos, on="nome", how="inner")
df_merge = df_merge.drop(columns=['turma_sheet'])
df_merge

,nome,nota_final,turma,classroom_id,discipline_id,aluno_id,numero,condicao,classroom_evaluation_id,avaliacao_nome
0,ALICE VITORIA FALCAO DE JESUS,5.0,1ªV01-EM-LCH,36381,261719,845461,1,active,1739069,LISTAS DE EXERCICIOS
1,ALICE VITORIA FALCAO DE JESUS,5.0,1ªV01-EM-LCH,36381,261719,845461,1,active,1739082,PROVA
2,ALICE VITORIA FALCAO DE JESUS,5.0,1ªV01-EM-LCH,36381,261719,845461,1,active,1793318,TRABALHOS INDIVIDUAIS
3,ALICE VITORIA FALCAO DE JESUS,5.0,1ªV01-EM-LCH,36381,261719,845461,1,active,1910164,INTERDISCIPLINAR
4,ALICIA DOS SANTOS ROCHA,4.0,1ªV01-EM-LCH,36381,261719,793002,2,active,1739069,LISTAS DE EXERCICIOS
...,...,...,...,...,...,...,...,...,...,...
391,KAUA SILVA MONTEIRO,NaN,3ªV01-EM-ENE,36378,257667,917395,29,active,1762542,PROVA
392,KAUA SILVA MONTEIRO,NaN,3ªV01-EM-ENE,36378,257667,917395,29,active,1762566,TRABALHOS EM GRUPOS
393,KAUA SILVA MONTEIRO,NaN,3ªV01-EM-ENE,36378,257667,917395,29,active,1762553,TRABALHOS INDIVIDUAIS
394,KAUA SILVA MONTEIRO,NaN,3ªV01-EM-ENE,36378,257667,917395,29,active,1762532,LISTAS DE EXERCICIOS


In [ ]:
df_inter = df_merge[
    df_merge["avaliacao_nome"].str.strip().str.upper() == "INTERDISCIPLINAR"
]
df_inter

,nome,nota_final,turma,classroom_id,discipline_id,aluno_id,numero,condicao,classroom_evaluation_id,avaliacao_nome
3,ALICE VITORIA FALCAO DE JESUS,5.0,1ªV01-EM-LCH,36381,261719,845461,1,active,1910164,INTERDISCIPLINAR
7,ALICIA DOS SANTOS ROCHA,4.0,1ªV01-EM-LCH,36381,261719,793002,2,active,1910164,INTERDISCIPLINAR
11,ARTHUR GOMES SPAGNOL,2.0,1ªV01-EM-LCH,36381,261719,821774,3,active,1910164,INTERDISCIPLINAR
15,AUGUSTO DOS SANTOS GERONIMO,2.0,1ªV01-EM-LCH,36381,261719,896039,4,active,1910164,INTERDISCIPLINAR
19,CAMILLY VICTORIA GUIMARAES DA SILVA LOURENCO,2.0,1ªV01-EM-LCH,36381,261719,806212,5,active,1910164,INTERDISCIPLINAR
...,...,...,...,...,...,...,...,...,...,...
375,VICTOR GABRIEL SILVA NOVAIS,NaN,3ªV01-EM-ENE,36378,257667,724678,25,active,1910162,INTERDISCIPLINAR
380,WILLIANY ALMEIDA SEPULCHRO,NaN,3ªV01-EM-ENE,36378,257667,863752,26,active,1910162,INTERDISCIPLINAR
385,YASMYN ALVES FERREIRA,4.0,3ªV01-EM-ENE,36378,257667,724681,27,active,1910162,INTERDISCIPLINAR
390,GUILHERME DE MATOS CAMILO,NaN,3ªV01-EM-ENE,36378,257667,916763,28,active,1910162,INTERDISCIPLINAR


In [ ]:
payloads = []

stage_id = int(minhas_turmas['stage_id'].iloc[0])

for _, row in df_inter.iterrows():

    nota = row["nota_final"]

    # 🔥 regra principal
    if pd.isna(nota):
        number = None
        unevaluated = True
    else:
        number = float(nota)
        unevaluated = False

    payload = {
        "classroom_evaluation_id": int(row["classroom_evaluation_id"]),
        "classroom_id": int(row["classroom_id"]),
        "curriculum_discipline_id": int(row["discipline_id"]),
        "group_student_id": int(row["aluno_id"]),

        "letter_id": None,

        "number": number,
        "recovery": None,

        "recovery_unevaluated": False,
        "stage_id": stage_id,  # ⚠️ ajuste se precisar (ou puxa do dataset)
        "unevaluated": unevaluated
    }

    payloads.append(payload)

In [ ]:
for p in payloads:
    seges.alterar_nota(p)

Aluno ID: 845461
Status: 200
Aluno ID: 793002
Status: 200
Aluno ID: 821774
Status: 200
Aluno ID: 896039
Status: 200
Aluno ID: 806212
Status: 200
Aluno ID: 834971
Status: 200
Aluno ID: 898898
Status: 200
Aluno ID: 865981
Status: 200
Aluno ID: 799012
Status: 200
Aluno ID: 809256
Status: 200
Aluno ID: 797287
Status: 200
Aluno ID: 843263
Status: 200
Aluno ID: 882173
Status: 200
Aluno ID: 819756
Status: 200
Aluno ID: 838684
Status: 200
Aluno ID: 881045
Status: 200
Aluno ID: 860541
Status: 200
Aluno ID: 890775
Status: 200
Aluno ID: 830390
Status: 200
Aluno ID: 874275
Status: 200
Aluno ID: 816138
Status: 200
Aluno ID: 852979
Status: 200
Aluno ID: 821600
Status: 200
Aluno ID: 879616
Status: 200
Aluno ID: 913165
Status: 200
Aluno ID: 914801
Status: 200
Aluno ID: 925679
Status: 200
Aluno ID: 918802
Status: 200
Aluno ID: 814656
Status: 200
Aluno ID: 815189
Status: 200
Aluno ID: 841171
Status: 200
Aluno ID: 840327
Status: 200
Aluno ID: 861732
Status: 200
Aluno ID: 917699
Status: 200
Aluno ID: 8005